# Transfer Trumans To AMASS Format

In [ ]:
# -*- coding: utf-8 -*-
import os

import numpy as np


# === Paths ===
base_path = "./data/Data_release_Trumans"
output_dir = "./data/Trumans_Motion_Processed/npz_y_up"
os.makedirs(output_dir, exist_ok=True)


# === Load global data ===
seg_names = np.load(
    os.path.join(base_path, "seg_name.npy"),
    allow_pickle=True,
)
human_pose = np.load(
    os.path.join(base_path, "human_pose.npy"),
    mmap_mode="r",
)
human_orient = np.load(
    os.path.join(base_path, "human_orient.npy"),
    mmap_mode="r",
)
human_transl = np.load(
    os.path.join(base_path, "human_transl.npy"),
    mmap_mode="r",
)
lhand_pose = np.load(
    os.path.join(base_path, "left_hand_pose.npy"),
    mmap_mode="r",
)
rhand_pose = np.load(
    os.path.join(base_path, "right_hand_pose.npy"),
    mmap_mode="r",
)

betas_all = np.load(
    os.path.join(base_path, "betas.npy"),
    allow_pickle=True,
)
meta = np.load(
    os.path.join(base_path, "meta.npy"),
    allow_pickle=True,
)

if isinstance(meta, np.ndarray) and meta.shape == ():
    meta = meta.item()  # Convert a scalar ndarray to a dictionary
elif isinstance(meta, dict):
    pass
else:
    meta = {}


# === Extract unique motion IDs ===
unique_segs = np.unique(seg_names)
print("Total number of motion segments:", len(unique_segs))


# === Process each motion segment ===
for i, seg_id in enumerate(unique_segs):
    seg_idx = np.where(seg_names == seg_id)[0]

    transl = human_transl[seg_idx]  # (N, 3)
    orient = human_orient[seg_idx]  # (N, 3)
    pose = human_pose[seg_idx]      # (N, 63)
    lhand = lhand_pose[seg_idx]     # (N, 45)
    rhand = rhand_pose[seg_idx]     # (N, 45)

    # Concatenate the full pose vector: (N, 156)
    poses = np.concatenate(
        [orient, pose, lhand, rhand],
        axis=1,
    )

    # Betas and gender are assumed to remain unchanged across frames
    betas = betas_all[seg_idx[0]]
    gender = meta.get("gender", "neutral")

    # Fixed mocap frame rate
    mocap_framerate = 30

    # Save as an NPZ file
    out_file = os.path.join(output_dir, f"{seg_id}.npz")
    np.savez(
        out_file,
        trans=transl,
        gender=gender,
        mocap_framerate=mocap_framerate,
        betas=betas,
        poses=poses,
    )

    if i % 50 == 0:
        print(f"[{i}/{len(unique_segs)}] Saved {out_file}")

print("All motion segments have been saved successfully.")

# Transfer Y-UP To Z-UP

In [ ]:
# -*- coding: utf-8 -*-
import os

import numpy as np
from scipy.spatial.transform import Rotation as R


IN_DIR = "./data/Trumans_Motion_Processed/npz_y_up"
OUT_DIR = "./data/Trumans_Motion_Processed/npz_z_up"
os.makedirs(OUT_DIR, exist_ok=True)


# Y-up -> Z-up: active rotation of -90 degrees around the X-axis
R_YUP_TO_ZUP = np.array(
    [
        [1, 0, 0],
        [0, 0, -1],
        [0, 1, 0],
    ],
    dtype=np.float32,
)


def yup_to_zup_trans(trans):
    return (trans @ R_YUP_TO_ZUP.T).astype(np.float32) + np.array([0.0, +0.2, -0.4], dtype=np.float32)


def yup_to_zup_root_orient_axis_angle(orient_axis_angle):
    aa = orient_axis_angle.reshape(-1, 3)

    # Convert axis-angle rotations to rotation matrices
    rotm = R.from_rotvec(aa).as_matrix()

    # Transform the global orientation from Y-up to Z-up
    rotm_zup = R_YUP_TO_ZUP @ rotm

    # Convert rotation matrices back to axis-angle
    aa_zup = R.from_matrix(rotm_zup).as_rotvec()
    aa_zup = aa_zup.reshape(orient_axis_angle.shape)

    return aa_zup.astype(np.float32)


def process_file(in_file, out_file):
    data = np.load(in_file, allow_pickle=True)

    trans = data["trans"]
    poses = data["poses"]
    gender = data["gender"]
    mocap_framerate = data["mocap_framerate"]
    betas = data["betas"]

    assert poses.shape[1] >= 3, "poses must contain global_orient (3 values)"

    # Coordinate transformation
    trans_zup = yup_to_zup_trans(trans)

    orient_yup = poses[:, :3]
    orient_zup = yup_to_zup_root_orient_axis_angle(orient_yup)

    poses_zup = poses.copy().astype(np.float32)
    poses_zup[:, :3] = orient_zup

    # Save the converted data
    np.savez(
        out_file,
        trans=trans_zup.astype(np.float32),
        gender=gender,
        mocap_framerate=int(mocap_framerate),
        betas=betas.astype(np.float32),
        poses=poses_zup.astype(np.float32),
    )

    print("Converted:", os.path.basename(out_file))


def main():
    files = [f for f in os.listdir(IN_DIR) if f.endswith(".npz")]
    print(f"Found {len(files)} files")

    for i, fname in enumerate(files, start=1):
        in_file = os.path.join(IN_DIR, fname)
        out_file = os.path.join(OUT_DIR, fname)

        process_file(in_file, out_file)

        if i % 50 == 0 or i == len(files):
            print(f"[{i}/{len(files)}] Processing...")

    print("All files have been converted.")
    print("Output directory:", OUT_DIR)


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
import os

import numpy as np


MOTION_DIR = "./data/Trumans_Motion_Processed/npz_z_up"
OUT_DIR = "./data/Trumans_Motion_Processed/npz_z_up_split_121"

FIXED_LEN = 121

os.makedirs(OUT_DIR, exist_ok=True)

total_saved = 0
total_skipped = 0

all_files = sorted(
    f for f in os.listdir(MOTION_DIR) if f.endswith(".npz")
)

print(f"[INFO] Found {len(all_files)} motion files")

for fname in all_files:
    base = fname[:-4]
    motion_path = os.path.join(MOTION_DIR, fname)

    print(f"\n[INFO] Processing motion: {base}")

    data = np.load(motion_path, allow_pickle=True)
    arrays = {k: data[k] for k in data.files}

    # Determine the temporal length
    if "trans" in arrays:
        T = arrays["trans"].shape[0]
    else:
        T = None
        for value in arrays.values():
            if isinstance(value, np.ndarray) and value.ndim > 0:
                T = value.shape[0]
                break

        if T is None:
            print(f"[WARN] Temporal dimension not found. Skipping {base}")
            total_skipped += 1
            continue

    print(f"[INFO]   T = {T}")

    num_segments = T // FIXED_LEN

    if num_segments == 0:
        print(f"[WARN]   Motion is shorter than {FIXED_LEN} frames. Skipping")
        total_skipped += 1
        continue

    for i in range(num_segments):
        start = i * FIXED_LEN
        end = start + FIXED_LEN - 1  # Inclusive end index

        out_dict = {}

        for key, value in arrays.items():
            if (
                isinstance(value, np.ndarray)
                and value.ndim > 0
                and value.shape[0] == T
            ):
                out_dict[key] = value[start:end + 1]
            else:
                out_dict[key] = value

        # Segment metadata
        out_dict["seg_start_incl"] = np.int32(start)
        out_dict["seg_end_incl"] = np.int32(end)
        out_dict["seg_length"] = np.int32(FIXED_LEN)
        out_dict["action_text"] = np.array("fixed_interval")

        out_name = f"{base}_{start:05d}_{end:05d}.npz"
        out_path = os.path.join(OUT_DIR, out_name)

        np.savez_compressed(out_path, **out_dict)
        print(f"[SAVE]   {out_name}")

        total_saved += 1

print("\n[INFO] Processing completed")
print(f"[INFO] Total saved segments: {total_saved}")
print(f"[INFO] Total skipped motions: {total_skipped}")

# Filter Bad Frames

In [ ]:
#!/usr/bin/env python3
import os
import numpy as np
from tqdm import tqdm


DATA_ROOT = "./data"

BAD_FRAMES_PATH = os.path.join(
    DATA_ROOT,
    "Data_release_trumans/bad_frames.npy"
)
SEG_NAME_PATH = os.path.join(
    DATA_ROOT,
    "Data_release_trumans/seg_name.npy"
)
FRAME_ID_PATH = os.path.join(
    DATA_ROOT,
    "Data_release_trumans/frame_id.npy"
)

IN_DIR  = "./data/Trumans_Motion_Processed/npz_z_up_split_121"
OUT_DIR = "./data/Trumans_Motion_Processed/npz_z_up_split_121_clean"

os.makedirs(OUT_DIR, exist_ok=True)


bad_frames = np.load(BAD_FRAMES_PATH)
seg_name   = np.load(SEG_NAME_PATH)
frame_id   = np.load(FRAME_ID_PATH)

print(f"[INFO] total global frames = {len(seg_name)}")
print(f"[INFO] bad frames count   = {len(bad_frames)}")


bad_map = {}

for gidx in bad_frames:
    gidx = int(gidx)
    seg  = str(seg_name[gidx])
    fid  = int(frame_id[gidx])
    bad_map.setdefault(seg, set()).add(fid)

print(f"[INFO] bad segments (unique) = {len(bad_map)}")


kept = 0
removed = 0

all_files = sorted(f for f in os.listdir(IN_DIR) if f.endswith(".npz"))

for fname in tqdm(all_files, desc="Filtering"):
    path = os.path.join(IN_DIR, fname)

    base = fname[:-4]
    motion_base, start_str, end_str = base.rsplit("_", 2)

    start = int(start_str)
    end   = int(end_str)

    if motion_base not in bad_map:
        os.symlink(path, os.path.join(OUT_DIR, fname))
        kept += 1
        continue

    bad_fids = bad_map[motion_base]

    has_bad = any((start <= fid <= end) for fid in bad_fids)

    if has_bad:
        removed += 1
    else:
        os.symlink(path, os.path.join(OUT_DIR, fname))
        kept += 1

print("\n[INFO] Filtering done")
print(f"[INFO] kept segments    = {kept}")
print(f"[INFO] removed segments = {removed}")
print(f"[INFO] clean data dir   = {OUT_DIR}")


# Sample Test Data

In [ ]:
import os
import random
import shutil

source = "./data/Trumans_Motion_Processed/npz_z_up_split_121_clean"
train_dir = source + "_train"
inference_dir = source + "_inference"

os.makedirs(train_dir, exist_ok=True)
os.makedirs(inference_dir, exist_ok=True)

files = [f for f in os.listdir(source) if f.endswith(".npz")]

random.seed(42)
random.shuffle(files)

split = int(len(files) * 0.9)

for i, file in enumerate(files):
    target = train_dir if i < split else inference_dir
    shutil.move(
        os.path.join(source, file),
        os.path.join(target, file),
    )

print(f"Train: {split}")
print(f"Inference: {len(files) - split}")

# Convert to PKL

In [ ]:
!python ./scripts/data_process/convert_amass_data.py \
  --path ./data/Trumans_Motion_Processed/npz_z_up_split_121_clean_train \
  --debug \
  --upright_start\
  --save_path ./data/trumans_motion_train.pkl

In [ ]:
!python ./scripts/data_process/convert_amass_data.py \
  --path ./data/Trumans_Motion_Processed/npz_z_up_split_121_clean_inference \
  --debug \
  --upright_start\
  --save_path ./data/trumans_motion_inference.pkl